# CNN_5_7ch_Exp22 Bias Adjustment ของคลาส N2 และ R ใน Hard Voting Ensemble

# 1.Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

# 2. Setting

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")

EXP13_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed44",
}
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp22_Bias_N2_R"
assert RESULT_DIR not in EXP13_DIRS.values()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
PROB_COLS = ["prob_" + s for s in STAGES]
KEY_COLS = ["subject", "epoch_start", "epoch_end", "actual"]
SEEDS = [42, 43, 44]
SELECTED_BIAS = None


# 3.โหลด CSV และตรวจความตรงกัน

In [ ]:


saved = {
    seed: pd.read_csv(EXP13_DIRS[seed] / "val_predictions.csv")
    for seed in SEEDS
}
base = saved[42]
assert len(base) == 2878

for seed, df in saved.items():
    assert len(df) == len(base)
    for col in KEY_COLS:
        assert (df[col].astype(str).to_numpy() == base[col].astype(str).to_numpy()).all(), f"Seed {seed}: {col} ไม่ตรงกัน"

    prob = df[PROB_COLS].to_numpy()
    pred = np.array(STAGES)[prob.argmax(axis=1)]

    assert np.isfinite(prob).all()
    assert np.allclose(prob.sum(axis=1), 1, atol=1e-3)
    assert (pred == df["predicted"].to_numpy()).all()
    assert np.allclose(df["confidence"], prob.max(axis=1), atol=1e-5)
    assert (df["correct"].astype(str).str.upper().to_numpy() == np.where(df["actual"] == df["predicted"], "TRUE", "FALSE")).all()

y_val = base["actual"].map(dict(zip(STAGES, LABELS))).to_numpy()
assert not pd.isna(y_val).any()
y_val = y_val.astype(int)
probs = {seed: df[PROB_COLS].to_numpy() for seed, df in saved.items()}

print("ตัวอย่างตรงกัน:", len(y_val), "sequences,", base["subject"].nunique(), "subjects")

# 4. Soft Voting

In [ ]:
mean_probs = np.mean(np.stack([probs[s] for s in SEEDS], axis=0), axis=0)

def soft_vote_with_bias(bias_n2=0.0, bias_r=0.0):
    scores = np.log(np.clip(mean_probs, 1e-12, 1.0))
    scores[:, 2] += bias_n2
    scores[:, 4] += bias_r
    return scores.argmax(axis=1)

In [ ]:
def evaluate(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-Precision": p.mean(),
        "Macro-Recall": r.mean(),
        "Macro-F1": f.mean(),
        "Kappa": cohen_kappa_score(y_true, y_pred),
    }
    for i, s in enumerate(STAGES):
        out[f"{s}_Precision"] = p[i]
        out[f"{s}_Recall"] = r[i]
        out[f"{s}_F1"] = f[i]
    return out

# 5.ตรวจผล Hard Voting เดิม

In [ ]:
soft_pred = soft_vote_with_bias(0.0, 0.0)
baseline = evaluate(y_val, soft_pred)

print("Exp13 Soft Voting (Mean Probability)")
display(pd.DataFrame([baseline]).round(4))

pd.DataFrame([baseline]).to_csv(RESULT_DIR / "baseline_soft_voting.csv", index=False)

cm = confusion_matrix(y_val, soft_pred, labels=LABELS)
pd.DataFrame(cm, index=[f"Actual_{s}" for s in STAGES],
             columns=[f"Predicted_{s}" for s in STAGES]).to_csv(
    RESULT_DIR / "baseline_confusion_matrix.csv"
)

# 6. ตรวจว่า Bias = 0 ตรงกับ Baseline

In [ ]:
assert (soft_pred == mean_probs.argmax(axis=1)).all()
print("Baseline ตรงกับ Soft Voting (bias = 0)")

# 7. Bias Search ทีละคลาสบน Validation

In [ ]:
bias_values = np.round(np.arange(-1.0, 1.01, 0.25), 2)

results = []

for bias_n2 in bias_values:
    pred = soft_vote_with_bias(bias_n2=bias_n2, bias_r=0.0)
    row = {"Target": "N2", "Bias_N2": bias_n2, "Bias_R": 0.0}
    row.update(evaluate(y_val, pred))
    results.append(row)

for bias_r in bias_values:
    pred = soft_vote_with_bias(bias_n2=0.0, bias_r=bias_r)
    row = {"Target": "R", "Bias_N2": 0.0, "Bias_R": bias_r}
    row.update(evaluate(y_val, pred))
    results.append(row)

results_df = pd.DataFrame(results)

display(results_df[
    ["Target", "Bias_N2", "Bias_R", "Accuracy",
     "Macro-F1", "Kappa", "W_F1", "N1_F1",
     "N2_F1", "N3_F1", "R_F1"]
].round(4))

results_df.to_csv(RESULT_DIR / "validation_bias_single_class_results.csv",index=False)

# 7.1 คัดเลือก Bias จาก Validation ตามเกณฑ์ที่กำหนด

In [ ]:
eligible = results_df[
    (results_df["Macro-F1"] >= baseline["Macro-F1"] + 0.005) &
    (results_df["Kappa"]    >= baseline["Kappa"]    - 0.005) &
    (results_df["Accuracy"] >= baseline["Accuracy"] - 0.005) &
    (results_df["W_F1"]  >= baseline["W_F1"]  - 0.02) &
    (results_df["N1_F1"] >= baseline["N1_F1"] - 0.02) &
    (results_df["N2_F1"] >= baseline["N2_F1"] - 0.02) &
    (results_df["N3_F1"] >= baseline["N3_F1"] - 0.02) &
    (results_df["R_F1"]  >= baseline["R_F1"]  - 0.02)
].copy()

if eligible.empty:
    SELECTED_BIAS_N2, SELECTED_BIAS_R = 0.0, 0.0
    print("ไม่มี Bias ที่ผ่านเกณฑ์ คงใช้ Baseline")
else:
    eligible["Total_Bias"] = eligible["Bias_N2"].abs() + eligible["Bias_R"].abs()
    eligible = eligible.sort_values(
        ["Macro-F1", "Total_Bias"],
        ascending=[False, True]
    )
    SELECTED_BIAS_N2 = float(eligible.iloc[0]["Bias_N2"])
    SELECTED_BIAS_R = float(eligible.iloc[0]["Bias_R"])
    print("Selected Bias N2:", SELECTED_BIAS_N2)
    print("Selected Bias R:", SELECTED_BIAS_R)

# 8. กราฟผลตามค่า Bias

In [ ]:
plt.figure(figsize=(9, 5))

for col, label in [
    ("Macro-F1", "Macro-F1"),
    ("N2_F1", "N2 F1"),
    ("R_F1", "R F1"),
    ("Accuracy", "Accuracy")
]:
    plt.plot(range(len(results_df)), results_df[col], marker="o", label=label)

plt.xlabel("Bias Combination")
plt.ylabel("Score")
plt.title("Exp22: Bias Adjustment on Validation")
plt.xticks(range(len(results_df)),
           [f"{n2:.2f},{r:.2f}" for n2, r in zip(results_df["Bias_N2"], results_df["Bias_R"])],
           rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULT_DIR / "validation_bias_plot.png", dpi=300, bbox_inches="tight")
plt.show()


# 9. บันทึกผล Bias ที่เลือกจาก Validation

In [ ]:
selected_pred = soft_vote_with_bias(SELECTED_BIAS_N2, SELECTED_BIAS_R)
selected_metrics = evaluate(y_val, selected_pred)

print("Selected Bias N2:", SELECTED_BIAS_N2)
print("Selected Bias R:", SELECTED_BIAS_R)
display(pd.DataFrame([selected_metrics]).round(4))

pd.DataFrame([{
    "Selected_Bias_N2": SELECTED_BIAS_N2,
    "Selected_Bias_R": SELECTED_BIAS_R,
    **selected_metrics
}]).to_csv(RESULT_DIR / "selected_bias_validation_metrics.csv", index=False)

pd.DataFrame({
    "subject": base["subject"],
    "epoch_start": base["epoch_start"],
    "epoch_end": base["epoch_end"],
    "actual": base["actual"],
    "predicted": np.array(STAGES)[selected_pred],
    "correct": selected_pred == y_val
}).to_csv(RESULT_DIR / "selected_bias_validation_predictions.csv", index=False)

cm = confusion_matrix(y_val, selected_pred, labels=LABELS)
pd.DataFrame(cm, index=[f"Actual_{s}" for s in STAGES],
             columns=[f"Predicted_{s}" for s in STAGES]).to_csv(
    RESULT_DIR / "selected_bias_confusion_matrix.csv"
)

print("บันทึกผล Exp22 เรียบร้อย")
